# Fake News Detection

**Dataset:** ISOT Fake News detection dataset (binary text classification) (`Fake.csv = 0` | `True.csv = 1`)    
**Source:** https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

Project by: Sania       
CMS ID: 023-24-0277

## 0. Installation

In [ ]:
!pip install -q nltk scikit-learn pandas numpy matplotlib seaborn scipy

## 1. Download Dataset 
Dataset separated in two files:

Fake.csv (23502 fake news article)    
True.csv (21417 true news article)

## 2. Imports and NLTK Resource Setup

In [ ]:
import re
import time
import warnings
import numpy as np
import pandas as pd
import nltk

warnings.filterwarnings('ignore')

from collections import Counter

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from scipy.sparse import issparse

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

# Download required NLTK resources
for resource in [
    'punkt',
    'punkt_tab',
    'stopwords',
    'wordnet',
    'averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng'
]:
    nltk.download(resource, quiet=True)

print("All imports and NLTK resources ready.")

## 3. Data Loading and EDA

In [ ]:
fake = pd.read_csv('Fake.csv')
real = pd.read_csv('True.csv')

fake['label'] = 0
real['label'] = 1

print("Fake articles:", len(fake))
print("Real articles:", len(real))

# Balanced sampling for faster experimentation
N = 5000

df = pd.concat([
    fake.sample(min(N, len(fake)), random_state=42),
    real.sample(min(N, len(real)), random_state=42)
], ignore_index=True)

df = df[['text', 'label']].dropna(subset=['text'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != '']

# Remove duplicate articles
df = df.drop_duplicates(subset=['text'])
df = df.reset_index(drop=True)

print("\nWorking dataset:", len(df))
print(df['label'].value_counts().rename({
    0: 'Fake',
    1: 'Real'
}))

In [ ]:
# Basic EDA
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(
    axes,
    ['word_count', 'char_count'],
    ['Word Count Distribution', 'Character Count Distribution']
):
    for label, name in [(0, 'Fake'), (1, 'Real')]:
        subset = df[df['label'] == label][col]
        ax.hist(
            subset.clip(upper=subset.quantile(0.99)),
            bins=60,
            alpha=0.6,
            label=name
        )
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()

print(
    df.groupby('label')[['word_count', 'char_count']]
    .describe()
    .round(1)
)

## 4. Text Preprocessing

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

# Preserve negation words because they can change meaning.
NEGATION_KEEP = {
    'no', 'not', 'never', 'neither',
    'nor', 'nobody', 'nothing', 'nowhere'
}

STOP_WORDS -= NEGATION_KEEP

lemmatizer = WordNetLemmatizer()


def preprocess_for_tfidf(text):
    """
    Text preprocessing:
    1. Lowercase
    2. Remove URLs
    3. Remove numbers
    4. Remove special characters
    5. Tokenization
    6. Stopword removal
    7. Lemmatization
    """
    text = str(text).lower()

    text = re.sub(
        r'https?://\\S+|www\\.\\S+',
        ' ',
        text
    )
    text = re.sub(r'\\d+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha()
        and (
            token not in STOP_WORDS
            or token in NEGATION_KEEP
        )
    ]

    return ' '.join(tokens)


print("Preprocessing example:")
print("\nOriginal:")
print(df['text'].iloc[0][:200])
print("\nCleaned:")
print(preprocess_for_tfidf(df['text'].iloc[0])[:200])

# IMPORTANT:
# Do not calculate common_high_freq on the complete dataset.
# TF-IDF will learn its vocabulary from training folds only.

## 5. Linguistic and Syntactic Feature Engineering

In [ ]:
def get_pos_tags(text):
    tokens = word_tokenize(str(text)[:3000])
    return pos_tag(tokens)


def superlative_ratio(tags):
    total = len(tags)
    if total == 0:
        return 0.0
    return sum(1 for _, tag in tags if tag in ('JJS', 'RBS')) / total


def proper_noun_ratio(tags):
    total = len(tags)
    if total == 0:
        return 0.0
    return sum(1 for _, tag in tags if tag in ('NNP', 'NNPS')) / total


def first_person_ratio(tags):
    total = len(tags)
    if total == 0:
        return 0.0
    first_person = {
        'i', 'me', 'my', 'myself',
        'we', 'us', 'our', 'ours', 'ourselves'
    }
    return sum(
        1 for word, tag in tags
        if tag.startswith('PRP') and word.lower() in first_person
    ) / total


def pronoun_to_noun_ratio(tags):
    pron = sum(1 for _, tag in tags if tag.startswith('PRP'))
    nouns = sum(1 for _, tag in tags if tag.startswith('NN'))
    denominator = pron + nouns
    return pron / denominator if denominator else 0.0


def modal_verb_ratio(tags):
    total = len(tags)
    if total == 0:
        return 0.0
    return sum(1 for _, tag in tags if tag == 'MD') / total


def exclamation_density(text):
    sentences = sent_tokenize(str(text))
    return str(text).count('!') / max(len(sentences), 1)


def question_density(text):
    sentences = sent_tokenize(str(text))
    return str(text).count('?') / max(len(sentences), 1)


def avg_sentence_length(text):
    sentences = sent_tokenize(str(text))
    if not sentences:
        return 0.0
    return np.mean([len(word_tokenize(sentence)) for sentence in sentences])


def type_token_ratio(text):
    tokens = word_tokenize(str(text).lower())
    tokens = [token for token in tokens if token.isalpha()]
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def uppercase_word_ratio(text):
    words = str(text).split()
    if not words:
        return 0.0
    return sum(
        1 for word in words
        if word.isupper() and len(word) > 1
    ) / len(words)


LING_FEATURES = [
    'superlative_ratio',
    'proper_noun_ratio',
    'first_person_ratio',
    'pron_noun_ratio',
    'modal_ratio',
    'exclamation_den',
    'question_den',
    'avg_sent_len',
    'type_token_ratio',
    'uppercase_ratio'
]


def extract_linguistic_features(text):
    tags = get_pos_tags(text)
    return [
        superlative_ratio(tags),
        proper_noun_ratio(tags),
        first_person_ratio(tags),
        pronoun_to_noun_ratio(tags),
        modal_verb_ratio(tags),
        exclamation_density(text),
        question_density(text),
        avg_sentence_length(text),
        type_token_ratio(text),
        uppercase_word_ratio(text)
    ]


# Transformer allows linguistic extraction inside CV pipelines.
class LinguisticExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray([
            extract_linguistic_features(text)
            for text in X
        ], dtype=float)

In [ ]:
# Extract features for exploratory analysis only.
# These are not used to fit the final model outside its pipeline.
print("Extracting linguistic features for EDA...")

t0 = time.time()

ling_df = pd.DataFrame(
    [
        extract_linguistic_features(text)
        for text in df['text']
    ],
    columns=LING_FEATURES
)

print("Feature extraction completed in:", round(time.time() - t0, 1), "seconds")

print("\nFeature statistics by label:")
print(
    ling_df.assign(label=df['label'].values)
    .groupby('label')[LING_FEATURES]
    .mean()
    .T
    .rename(columns={0: 'Fake', 1: 'Real'})
    .round(4)
)

# Visualize linguistic feature distributions
fig, axes = plt.subplots(2, 5, figsize=(20, 7))

for ax, feature in zip(axes.flatten(), LING_FEATURES):
    for label, name in [(0, 'Fake'), (1, 'Real')]:
        values = ling_df.loc[df['label'] == label, feature]
        ax.hist(
            values,
            bins=40,
            alpha=0.6,
            density=True,
            label=name
        )
    ax.set_title(feature, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle(
    'Linguistic Feature Distributions: Fake vs Real',
    fontsize=13
)
plt.tight_layout()
plt.show()

## 6. Train / Test Split

In [ ]:
X = df['text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training articles:", len(X_train))
print("Testing articles:", len(X_test))
print("Fake training percentage:", round((y_train == 0).mean() * 100, 2), "%")
print("Real training percentage:", round((y_train == 1).mean() * 100, 2), "%")

## 7. Leakage-Safe Feature Pipelines

In [ ]:
def build_pipeline(model, combined=True, nb=False):
    """
    TF-IDF and linguistic preprocessing are fitted
    independently inside every cross-validation fold.

    combined=True:
        TF-IDF + linguistic features

    combined=False:
        TF-IDF only

    nb=True:
        MinMax scaling for non-negative NB features.
    """
    tfidf = TfidfVectorizer(
        preprocessor=preprocess_for_tfidf,
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        max_features=50000,
        sublinear_tf=True,
        min_df=3,
        max_df=0.90
    )

    if combined:
        ling_pipeline = Pipeline([
            ('extract', LinguisticExtractor()),
            ('scale', MinMaxScaler() if nb else StandardScaler())
        ])

        features = FeatureUnion([
            ('tfidf', tfidf),
            ('linguistic', ling_pipeline)
        ])
    else:
        features = tfidf

    return Pipeline([
        ('features', features),
        ('classifier', model)
    ])

## 8. Cross-Validation and Evaluation Helpers

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_results = {}


def evaluate_model(name, model):
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    # Cross-validation on the COMPLETE pipeline.
    # Each fold learns its own vocabulary and scaling values.
    cv_start = time.time()
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # Fit final model using complete training data.
    fit_start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - fit_start

    # Evaluate on untouched test data.
    y_pred = model.predict(X_test)

    results = {
        'name': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
        'cv_time_s': cv_time,
        'train_time_s': fit_time,
        'y_pred': y_pred,
        'model': model
    }

    all_results[name] = results

    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"Test F1: {results['f1_weighted']:.4f}")
    print(
        f"5-Fold CV F1: "
        f"{results['cv_f1_mean']:.4f} "
        f"+/- {results['cv_f1_std']:.4f}"
    )
    print(f"CV time: {cv_time:.1f}s")
    print(f"Final training time: {fit_time:.1f}s")
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=['Fake', 'Real']
        )
    )

    return results

## 9. Experiment A: TF-IDF Only

In [ ]:
print("\nEXPERIMENT A: TF-IDF ONLY")

tfidf_models = {
    'LR (TF-IDF)': build_pipeline(
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ),
        combined=False
    ),
    'CNB (TF-IDF)': build_pipeline(
        ComplementNB(alpha=0.1),
        combined=False
    ),
    'SVM (TF-IDF)': build_pipeline(
        LinearSVC(
            C=0.5,
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ),
        combined=False
    ),
    'RF (TF-IDF)': build_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=42,
            n_jobs=1
        ),
        combined=False
    )
}

for name, model in tfidf_models.items():
    evaluate_model(name, model)

## 10. Experiment B: TF-IDF + Linguistic Features

In [ ]:
print("\nEXPERIMENT B: TF-IDF + LINGUISTIC FEATURES")

combined_models = {
    'LR (Combined)': build_pipeline(
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        )
    ),
    'CNB (Combined)': build_pipeline(
        ComplementNB(alpha=0.1),
        nb=True
    ),
    'SVM (Combined)': build_pipeline(
        LinearSVC(
            C=0.5,
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        )
    ),
    'RF (Combined)': build_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=42,
            n_jobs=1
        )
    )
}

for name, model in combined_models.items():
    evaluate_model(name, model)

## 11. Hyperparameter Tuning

In [ ]:
# 11A. LOGISTIC REGRESSION
print("\nHYPERPARAMETER TUNING")

lr_pipeline = build_pipeline(
    LogisticRegression(
        max_iter=1500,
        class_weight='balanced',
        random_state=42
    ),
    combined=False
)

lr_param_dist = {
    'classifier__C': [
        0.01, 0.05, 0.1, 0.5,
        1.0, 5.0, 10.0
    ],
    'classifier__penalty': [
        'l1', 'l2'
    ]
}

lr_search = RandomizedSearchCV(
    lr_pipeline,
    param_distributions=lr_param_dist,
    n_iter=10,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lr_search.fit(X_train, y_train)

print("\nBest LR parameters:")
print(lr_search.best_params_)
print("Best LR CV F1:")
print(lr_search.best_score_)

evaluate_model(
    'LR Tuned (TF-IDF)',
    lr_search.best_estimator_
)

In [ ]:
# 11B. LINEAR SVM
svm_pipeline = build_pipeline(
    LinearSVC(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ),
    combined=True
)

svm_param_dist = {
    'classifier__C': [
        0.01, 0.05, 0.1,
        0.5, 1.0, 2.0, 5.0
    ]
}

svm_search = RandomizedSearchCV(
    svm_pipeline,
    param_distributions=svm_param_dist,
    n_iter=7,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

svm_search.fit(X_train, y_train)

print("\nBest SVM parameters:")
print(svm_search.best_params_)
print("Best SVM CV F1:")
print(svm_search.best_score_)

evaluate_model(
    'SVM Tuned (Combined)',
    svm_search.best_estimator_
)

## 12. Confusion Matrices

In [ ]:
selected = [
    ('LR (TF-IDF)', all_results['LR (TF-IDF)']['y_pred']),
    ('LR (Combined)', all_results['LR (Combined)']['y_pred']),
    ('SVM (TF-IDF)', all_results['SVM (TF-IDF)']['y_pred']),
    ('SVM (Combined)', all_results['SVM (Combined)']['y_pred']),
    ('RF (TF-IDF)', all_results['RF (TF-IDF)']['y_pred']),
    ('RF (Combined)', all_results['RF (Combined)']['y_pred']),
    ('LR Tuned', all_results['LR Tuned (TF-IDF)']['y_pred']),
    ('SVM Tuned', all_results['SVM Tuned (Combined)']['y_pred'])
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, (name, y_pred) in zip(axes.flatten(), selected):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Fake', 'Real'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle('Confusion Matrices - All Models', fontsize=14)
plt.tight_layout()
plt.show()

## 13. Model Comparison Summary

In [ ]:
summary = []

for key, result in all_results.items():
    summary.append({
        'Model': key,
        'Accuracy': result['accuracy'],
        'F1 (weighted)': result['f1_weighted'],
        'CV F1 mean': result['cv_f1_mean'],
        'CV F1 std': result['cv_f1_std'],
        'Train time (s)': result['train_time_s'],
        'CV time (s)': result['cv_time_s']
    })

summary_df = pd.DataFrame(summary).set_index('Model').sort_values(
    'F1 (weighted)',
    ascending=False
)

print("\nMODEL PERFORMANCE COMPARISON")
print(summary_df.round(4).to_string())

# Performance chart
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(summary_df))
bar_width = 0.35

bars1 = ax.bar(
    x - bar_width / 2,
    summary_df['Accuracy'],
    width=bar_width,
    label='Accuracy',
    alpha=0.85
)

bars2 = ax.bar(
    x + bar_width / 2,
    summary_df['F1 (weighted)'],
    width=bar_width,
    label='F1 Weighted',
    alpha=0.85
)

ax.set_xticks(x)
ax.set_xticklabels(
    summary_df.index,
    rotation=35,
    ha='right',
    fontsize=9
)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend()

for bar in list(bars1) + list(bars2):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.005,
        f'{height:.3f}',
        ha='center',
        fontsize=7
    )

plt.tight_layout()
plt.show()

## 14. Ablation Analysis

In [ ]:
pairs = [
    ('Logistic Regression', 'LR (TF-IDF)', 'LR (Combined)'),
    ('Naive Bayes', 'CNB (TF-IDF)', 'CNB (Combined)'),
    ('Linear SVM', 'SVM (TF-IDF)', 'SVM (Combined)'),
    ('Random Forest', 'RF (TF-IDF)', 'RF (Combined)')
]

ablation_rows = []

for model_name, tfidf_key, combined_key in pairs:
    f1_tfidf = all_results[tfidf_key]['f1_weighted']
    f1_combined = all_results[combined_key]['f1_weighted']
    delta = f1_combined - f1_tfidf

    ablation_rows.append({
        'Model': model_name,
        'TF-IDF only F1': f1_tfidf,
        'TF-IDF + Linguistic F1': f1_combined,
        'Delta': delta,
        'Improvement': 'Yes' if delta > 0 else 'No'
    })

abl_df = pd.DataFrame(ablation_rows).set_index('Model')

print("\nABLATION ANALYSIS")
print(abl_df.round(4).to_string())

# Delta chart
fig, ax = plt.subplots(figsize=(8, 4))

colors = [
    '#2ecc71' if value >= 0 else '#e74c3c'
    for value in abl_df['Delta']
]

ax.bar(abl_df.index, abl_df['Delta'] * 100, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('F1 Change (Percentage Points)')
ax.set_title('Gain from Adding Linguistic Features')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 15. Top TF-IDF Features by Class

In [ ]:
# Retrieve the fitted TF-IDF vectorizer from the LR pipeline.
lr_fitted = all_results['LR (TF-IDF)']['model']
tfidf_fitted = lr_fitted.named_steps['features']
lr_fitted_classifier = lr_fitted.named_steps['classifier']

feature_names = np.array(tfidf_fitted.get_feature_names_out())
coefs = lr_fitted_classifier.coef_[0]

top_n = 20
real_indices = np.argsort(coefs)[-top_n:][::-1]
fake_indices = np.argsort(coefs)[:top_n]

top_real = feature_names[real_indices]
top_fake = feature_names[fake_indices]
top_real_values = coefs[real_indices]
top_fake_values = coefs[fake_indices]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(top_n), top_real_values[::-1], color='#2ecc71', alpha=0.85)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_real[::-1], fontsize=8)
axes[0].set_title('Top 20 Features → REAL News')
axes[0].set_xlabel('Logistic Regression Coefficient')

axes[1].barh(range(top_n), np.abs(top_fake_values[::-1]), color='#e74c3c', alpha=0.85)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top_fake[::-1], fontsize=8)
axes[1].set_title('Top 20 Features → FAKE News')
axes[1].set_xlabel('Absolute Logistic Regression Coefficient')

plt.suptitle('Most Discriminative TF-IDF Features', fontsize=13)
plt.tight_layout()
plt.show()

## 16. Linguistic Features Only

In [ ]:
# Linguistic-only pipeline.
# Scaling is also fitted inside each CV fold.
rf_ling_pipeline = Pipeline([
    ('extract', LinguisticExtractor()),
    ('scale', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    ))
])

evaluate_model('RF (Linguistic Only)', rf_ling_pipeline)

# Feature importance from fitted pipeline
rf_ling_fitted = all_results['RF (Linguistic Only)']['model']
rf_classifier = rf_ling_fitted.named_steps['classifier']

importances = pd.Series(
    rf_classifier.feature_importances_,
    index=LING_FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot.barh(ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Linguistic Feature Importance - Random Forest')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## 17. Optional: LSTM Sequence Model

In [ ]:
RUN_LSTM = False

if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import (
            Embedding,
            LSTM,
            Dense,
            Dropout,
            SpatialDropout1D
        )
        from tensorflow.keras.preprocessing.text import Tokenizer
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        from tensorflow.keras.callbacks import EarlyStopping
        TF_AVAILABLE = True
    except ImportError:
        TF_AVAILABLE = False
        print("TensorFlow not installed.")

    if TF_AVAILABLE:
        MAX_WORDS = 30000
        MAX_LEN = 300

        keras_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
        # Fit tokenizer on training data only.
        keras_tokenizer.fit_on_texts(X_train)

        X_seq_train = pad_sequences(
            keras_tokenizer.texts_to_sequences(X_train),
            maxlen=MAX_LEN,
            padding='post',
            truncating='post'
        )

        X_seq_test = pad_sequences(
            keras_tokenizer.texts_to_sequences(X_test),
            maxlen=MAX_LEN,
            padding='post',
            truncating='post'
        )

        lstm_model = Sequential([
            Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
            SpatialDropout1D(0.3),
            LSTM(64, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
            LSTM(32, dropout=0.2),
            Dense(32, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])

        lstm_model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        lstm_model.summary()

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )

        history = lstm_model.fit(
            X_seq_train,
            y_train,
            epochs=15,
            batch_size=64,
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=1
        )

        y_prob_lstm = lstm_model.predict(X_seq_test).flatten()
        y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

        lstm_accuracy = accuracy_score(y_test, y_pred_lstm)
        lstm_f1 = f1_score(y_test, y_pred_lstm, average='weighted')

        print("\nLSTM Accuracy:", round(lstm_accuracy, 4))
        print("LSTM Weighted F1:", round(lstm_f1, 4))
        print(classification_report(y_test, y_pred_lstm, target_names=['Fake', 'Real']))

        # Add LSTM to the comparison table.
        all_results['LSTM'] = {
            'name': 'LSTM',
            'accuracy': lstm_accuracy,
            'f1_weighted': lstm_f1,
            'cv_f1_mean': np.nan,
            'cv_f1_std': np.nan,
            'train_time_s': np.nan,
            'cv_time_s': np.nan,
            'y_pred': y_pred_lstm,
            'model': lstm_model
        }

        # Training curves
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, metric, title in zip(axes, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
            ax.plot(history.history[metric], label='Train')
            ax.plot(history.history[f'val_{metric}'], label='Validation')
            ax.set_title(f'LSTM {title}')
            ax.legend()
        plt.tight_layout()
        plt.show()

## 18. Final Model Comparison and Interpretation

In [ ]:
final_summary = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': result['accuracy'],
        'F1 (weighted)': result['f1_weighted'],
        'CV F1 mean': result['cv_f1_mean'],
        'CV F1 std': result['cv_f1_std']
    }
    for name, result in all_results.items()
]).set_index('Model').sort_values('F1 (weighted)', ascending=False)

print("\n" + "=" * 65)
print("FINAL MODEL COMPARISON - RANKED BY WEIGHTED F1")
print("=" * 65)
print(final_summary.round(4).to_string())

best_model_key = final_summary['F1 (weighted)'].idxmax()
best_f1 = final_summary['F1 (weighted)'].max()

print("\nBest model:", best_model_key)
print("Best weighted F1:", round(best_f1, 4))

print("\nKEY FINDINGS")
print("""
1. TF-IDF identifies important words and phrases
   that distinguish fake and real news articles.

2. Linguistic features provide additional information
   about writing style, sentence structure and word usage.

3. Combining TF-IDF with linguistic features allows
   comparison of lexical and stylistic information.

4. Logistic Regression and Linear SVM are useful
   linear models for high-dimensional text classification.

5. Complement Naive Bayes provides a probabilistic
   baseline for text classification.

6. Random Forest evaluates whether nonlinear
   relationships improve classification.

7. Cross-validation evaluates the complete feature
   pipeline, reducing data leakage.

8. The final test set is kept separate from model
   training and hyperparameter selection.

IMPORTANT:
High performance on this dataset does not establish
reliable detection of Pakistani fake news. The dataset
contains general Fake/Real news articles and may contain
source, topic and time-related biases.
""")